# TRICE — Business Entity Resolution (Kaggle runner)

**Team Vortex** · Aryan Bansal (lead) · Sachin Kumar · Daksh Tandon · Parth Aggarwal

Runs the full TRICE pipeline on Kaggle: prepare -> mine -> train -> tune -> infer, then
writes `matching_results.tsv` and `candidate_pairs.tsv` to `/kaggle/working/output/`.

## Before you run

1. **Add the code**: this notebook clones `https://github.com/iittjjee2024/TRICE.git`.
   No token needed for a public repo.
2. **Add the dataset**: *Add Input* -> *Datasets* -> your uploaded challenge dataset.
   Upload the seven TSVs preserving the `train/` and `test/` folders. The notebook
   auto-detects the mount under `/kaggle/input/`; if detection fails, set `DATA_DIR`
   manually in the config cell.
3. **Settings**: Accelerator *None* (CPU is fine; the models are gradient-boosted trees),
   Internet *On* (for the clone + pip). Persistence is optional.
4. Run all. On the full test set inference takes roughly 1.5–2.5 h; use the `SUBSET`
   knob in the config cell for a fast smoke run first.

In [8]:
# --- clone the code -------------------------------------------------------------
import os, subprocess, sys

REPO = "https://github.com/iittjjee2024/TRICE.git"
CODE = "/kaggle/working/TRICE"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=False)
print("code at", CODE)
print(sorted(os.listdir(CODE)))

Updating 1cfcead..c771f9d
Fast-forward
 kaggle/trice_kaggle_runner.ipynb |  7 ++--
 scripts/02_mine_variants.py      |  5 +--
 scripts/04_blocking_eval.py      |  2 +-
 scripts/test_encoding.py         | 69 ++++++++++++++++++++++++++++++++++++++++
 src/trice/evaluate.py            |  3 +-
 src/trice/export.py              |  4 +--
 src/trice/paths.py               | 23 +++++++++++++-
 src/trice/pipeline.py            |  3 +-
 src/trice/records.py             |  3 +-
 9 files changed, 108 insertions(+), 11 deletions(-)
 create mode 100644 scripts/test_encoding.py
code at /kaggle/working/TRICE
['.git', '.gitattributes', '.gitignore', '6ab5628d5a817_amazon_ml_challenge_problem_statement_copy.pdf', '6ab56657b4f1a_guidelines_and_key_instructions_amazon_ml_challenge_2026_copy.pdf', 'Documentation_template.md', 'LICENSE', 'README.md', 'backend', 'docs', 'frontend', 'kaggle', 'scripts', 'src', 'student_resource']


From https://github.com/iittjjee2024/TRICE
   1cfcead..c771f9d  main       -> origin/main


In [9]:
# --- dependencies ---------------------------------------------------------------
# Kaggle already ships numpy/pandas/scipy/scikit-learn/pyarrow/lightgbm. We only need
# the two light packages the pipeline adds. Versions are left to Kaggle's stack to avoid
# a slow, conflict-prone full reinstall.
!pip install -q rapidfuzz Unidecode
import numpy, pandas, sklearn, scipy, rapidfuzz, lightgbm, pyarrow, unidecode
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| sklearn", sklearn.__version__, "| lightgbm", lightgbm.__version__)

numpy 2.0.2 | pandas 2.3.3 | sklearn 1.6.1 | lightgbm 4.6.0


In [10]:
# --- configuration --------------------------------------------------------------
import glob, os

CODE = "/kaggle/working/TRICE"
WORK = "/kaggle/working/trice_run"       # writable: store, model, artifacts
OUT  = "/kaggle/working/output"          # the two submission TSVs land here
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# Fast smoke run vs full run. Set SUBSET=None for the real submission.
SUBSET = 4000        # entities/country for train; None = full
TRAIN_ENTITIES = 70000 if SUBSET is None else SUBSET

# If auto-detection fails, set this to the folder that directly contains the seven TSVs
# (or their train/ and test/ subfolders), e.g. "/kaggle/input/amazon-ml-2026".
DATA_DIR_OVERRIDE = None

# The seven files we need, keyed by the canonical name the pipeline expects. Kaggle
# both FLATTENS the train/ and test/ folders AND sometimes prefixes names with '_'
# (e.g. _train_source1.tsv), so we match each file by its distinctive STEM anywhere under
# /kaggle/input and ignore any leading punctuation.
REQUIRED = [
    ("train", "train_source1.tsv"), ("train", "train_source2.tsv"),
    ("train", "train_source3.tsv"), ("train", "train_ground_truth.tsv"),
    ("test", "test_source1.tsv"), ("test", "test_source2.tsv"),
    ("test", "test_source3.tsv"),
]

def _norm(name):
    """Lowercase basename with leading non-alphanumerics stripped ('_train..' -> 'train..')."""
    b = os.path.basename(name).lower()
    return b.lstrip("._- ")

def locate_files():
    """Return {(split, canonical_filename): absolute path}.

    Walks the whole /kaggle/input tree once and matches each .tsv by its normalised name,
    so a leading underscore or a flattened folder layout both resolve correctly.
    """
    all_tsv = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith(".tsv"):
                all_tsv.append(os.path.join(root, f))
    found = {}
    for split, fname in REQUIRED:
        target = fname.lower()
        hits = [p for p in all_tsv if _norm(p) == target]
        if hits:
            found[(split, fname)] = sorted(hits, key=len)[0]  # shallowest match
    return found

found = locate_files()
missing = [f"{s}/{f}" for (s, f) in REQUIRED if (s, f) not in found]

if missing:
    print("MISSING:", missing)
    print("\nEverything currently under /kaggle/input:")
    any_tsv = False
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        if os.path.isfile(p):
            print("  ", p)
            any_tsv = any_tsv or p.endswith(".tsv")
    if not any_tsv:
        print("  (no files at all — the dataset is not attached; use 'Add Input')")
    raise SystemExit(
        "Dataset files not found. Attach the challenge dataset via 'Add Input', or set "
        "DATA_DIR_OVERRIDE above to the folder shown in the listing.")

# Build a clean train/ + test/ layout in the writable working dir by symlinking the
# located files, so the pipeline sees exactly the structure it expects regardless of how
# the dataset was uploaded.
DATA_DIR = os.path.join(WORK, "dataset")
for split in ("train", "test"):
    os.makedirs(os.path.join(DATA_DIR, split), exist_ok=True)
for (split, fname), src in found.items():
    dst = os.path.join(DATA_DIR, split, fname)
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst)
    os.symlink(os.path.realpath(src), dst)

if DATA_DIR_OVERRIDE:
    DATA_DIR = DATA_DIR_OVERRIDE
print("DATA_DIR =", DATA_DIR)
for split in ("train", "test"):
    for fname in sorted(os.listdir(os.path.join(DATA_DIR, split))):
        p = os.path.join(DATA_DIR, split, fname)
        print(f"  {split}/{fname:<24} {os.path.getsize(p)/1e6:8.1f} MB")

DATA_DIR = /kaggle/working/trice_run/dataset
  train/train_ground_truth.tsv        0.0 MB
  train/train_source1.tsv             0.0 MB
  train/train_source2.tsv             0.0 MB
  train/train_source3.tsv             0.0 MB
  test/test_source1.tsv              0.0 MB
  test/test_source2.tsv              0.0 MB
  test/test_source3.tsv              0.0 MB


In [11]:
# --- wire the repo paths to the Kaggle layout -----------------------------------
# The pipeline scripts honour three environment variables (see src/trice/paths.py) so we
# point them at the writable Kaggle locations directly -- no symlinks, which used to be
# the fragile part. TRICE_DATA_DIR is the folder that holds train/ and test/.
import os

ART = os.path.join(WORK, "artifacts")
os.makedirs(ART, exist_ok=True)
os.environ["TRICE_DATA_DIR"] = DATA_DIR
os.environ["TRICE_ARTIFACTS_DIR"] = ART
os.environ["TRICE_OUTPUT_DIR"] = OUT

# sanity: confirm the resolver sees the files
import importlib, sys as _sys
_sys.path.insert(0, os.path.join(CODE, "src"))
from trice import paths as _paths          # noqa: E402
importlib.reload(_paths)
print("TRICE_DATA_DIR      =", _paths.dataset_dir(CODE))
print("TRICE_ARTIFACTS_DIR =", _paths.artifacts_dir(CODE))
print("TRICE_OUTPUT_DIR    =", _paths.output_dir(CODE))
gt = os.path.join(_paths.dataset_dir(CODE), 'train', 'train_ground_truth.tsv')
assert os.path.isfile(gt), f'ground truth not found at {gt}'
print('ground truth OK:', gt)

TRICE_DATA_DIR      = /kaggle/working/trice_run/dataset
TRICE_ARTIFACTS_DIR = /kaggle/working/trice_run/artifacts
TRICE_OUTPUT_DIR    = /kaggle/working/output
ground truth OK: /kaggle/working/trice_run/dataset/train/train_ground_truth.tsv


In [12]:
# --- correctness tests (fast, need no dataset) ----------------------------------
import subprocess
PY = sys.executable
for t in ("test_decide.py", "test_union.py"):
    print("=" * 70, "\n", t)
    subprocess.run([PY, os.path.join(CODE, "scripts", t)], cwd=CODE, check=True)

 test_decide.py
0. worked example from the problem statement
   predicted 3, truth 2, tp 2 -> F_0.5 = 0.714286  (statement says 0.714)

1. closed form vs precision/recall form, 20000 random set pairs
   max abs difference = 2.220e-16

2. Poisson-binomial pmf
   pmf matches brute force and sums to 1 for all cases

3. top-k prefix optimality vs exhaustive 2^n search
   400 random cases, expected-value gap max = 6.661e-16, mismatches = 0

4. behavioural sanity checks
   tiny probs      -> k*=0  E[F]=0.9653   (abstains)
   one strong      -> k*=1  E[F]=0.9668
   one @0.55       -> k*=1
   five @0.55      -> k*=5  E[F]=0.5938
   0.95 + 0.45     -> k*=1  E[F]=0.8787
   E[F|k] for [0.9,0.8,0.3,0.1] = k=0:0.0126, k=1:0.7346, k=2:0.8329, k=3:0.6994, k=4:0.5726  -> k*=2

5. k=0 expected value equals P(no true match)
   confirmed for all cases: E[F|0] = prod(1-p_i)

6. throughput
   20,000 entities in 7.67s = 2,609 entities/s
   -> 1,732,544 test entities would take ~664s

RESULT: all checks pass

In [15]:
# --- helper to stream a stage's output ------------------------------------------
# Streams the child process live AND keeps the last lines, so if a stage fails the real
# error (traceback) is reprinted right above the RuntimeError instead of being lost.
import subprocess, sys, time, collections
PY = sys.executable

def run(stage_args, title):
    print("=" * 78, f"\n{title}\n" + "=" * 78, flush=True)
    t0 = time.time()
    tail = collections.deque(maxlen=40)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen([PY, *stage_args], cwd=CODE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
        tail.append(line)
    p.wait()
    print(f"\n[{title}] exit={p.returncode}  {time.time()-t0:.0f}s", flush=True)
    if p.returncode != 0:
        print("\n----- last lines of failing stage -----")
        print("".join(tail))
        raise RuntimeError(f"{title} failed (exit {p.returncode}); see output above")

In [14]:
# --- preflight: run the pipeline setup IN-KERNEL so any error shows a full traceback
# (a subprocess would hide it). Reads the ground truth, normalises one record, checks
# the artifacts dir is writable. If this cell passes, the stages below will find their
# inputs.
import importlib
for _m in ("trice.paths", "trice.normalize", "trice.mine"):
    importlib.import_module(_m)
from trice.normalize import normalize_name, normalize_address
gt = os.path.join(DATA_DIR, "train", "train_ground_truth.tsv")
with open(gt, encoding="utf-8") as fh:
    header = next(fh)
    first = next(fh)
print("ground truth header:", header.rstrip().split("\t"))
print("first row:", first.rstrip()[:90], "...")
s1 = os.path.join(DATA_DIR, "train", "train_source1.tsv")
with open(s1, encoding="utf-8") as fh:
    next(fh)
    parts = next(fh).rstrip("\n").split("\t")
print("source1 columns:", len(parts), parts[:2])
nm = normalize_name(parts[1]); ad = normalize_address(parts[2])
print("normalise OK:", nm.core(), "|", ad.alpha_tokens[:4])
testfile = os.path.join(ART, "_write_test")
open(testfile, "w").close(); os.remove(testfile)
print("artifacts dir writable:", ART)
print("\nPREFLIGHT OK — stages below should run.")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xcc in position 37: invalid continuation byte

In [ ]:
# --- stage 1: mine token aliases from the training ground truth -----------------
run(["scripts/02_mine_variants.py", "--sample", "250000"], "mine variants")

In [ ]:
# --- stage 2: normalise all records into the Parquet record store ---------------
# Kaggle CPU has 4 cores; use 3 workers.
run(["scripts/03_prepare.py", "--workers", "3"], "prepare record store")

In [ ]:
# --- stage 3: train the matcher + score a held-out validation split -------------
run(["scripts/05_train.py", "--entities", str(TRAIN_ENTITIES), "--run-id", "kaggle"],
    "train + validate")

In [ ]:
# --- stage 4: search the decision-layer configuration on validation -------------
run(["scripts/07_tune_decision.py", "--run-id", "kaggle"], "tune decision")

In [ ]:
# --- inspect the validation metrics ---------------------------------------------
import json
m = json.load(open(os.path.join(ART, "runs", "kaggle", "metrics.json")))
print("model:", m["model_kind"], "| use_stage2:", m["use_stage2"])
print("stage1:", m["stage1"])
print("stage2:", m["stage2"])
ef = m["decision_rules"]["expected_f"]
print(f"VAL macro F0.5 = {ef['val_macro_f05']:.5f}  "
      f"P={ef['val_macro_precision']:.4f}  R={ef['val_macro_recall']:.4f}")
print("per-country:", m.get("decision_tuning", {}).get("by_country"))

In [ ]:
# --- stage 5: full test-set inference -> output/*.tsv ---------------------------
# Kaggle gives ~30 GB RAM (more than the 16 GB dev box), so the default query batch is
# safe; lower --query-batch if you hit memory limits.
run(["scripts/06_infer.py", "--run-id", "kaggle", "--query-batch", "120000"],
    "full test inference")

In [ ]:
# --- validate + preview the submittable TSV -------------------------------------
import subprocess
sub = subprocess.run(
    [PY, os.path.join(CODE, "student_resource/utils/validate_submission.py"),
     "--matching", os.path.join(OUT, "matching_results.tsv"),
     "--candidate", os.path.join(OUT, "candidate_pairs.tsv"),
     "--test-dir", os.path.join(DATA_DIR, "test")],
    cwd=CODE, capture_output=True, text=True)
print(sub.stdout)
print(sub.stderr)

print("=" * 60, "\nhead of matching_results.tsv:")
with open(os.path.join(OUT, "matching_results.tsv"), encoding="utf-8") as f:
    for i, line in zip(range(6), f):
        print(line.rstrip())

In [ ]:
# --- surface the outputs in /kaggle/working for download ------------------------
# Files under /kaggle/working are attached to the notebook's output; download
# matching_results.tsv from the Output tab and upload it to the challenge portal.
import shutil
for f in ("matching_results.tsv", "candidate_pairs.tsv"):
    src = os.path.join(OUT, f)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join("/kaggle/working", f))
        print(f"{f}: {os.path.getsize(src)/1e6:.1f} MB -> /kaggle/working/{f}")